# CSF-CUF · Interactive Quick Start

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/quickstart_colab_interactive.ipynb)

**Run a CUF analysis without writing Python or editing YAML files.** This notebook uses the existing [CSF-CUF Quick Start](https://github.com/giovanniboscu/continuous-section-field/tree/main/cuf/quickstart): **model** (geometry and materials), **problem** (load and boundary conditions), and **case** (CUF and longitudinal approximations).

**First time:** choose **Runtime → Run all**. Later, change any parameter in step 2 and run that cell again. The notebook automatically updates temporary YAML files, runs the solver and displays the response. The original GitHub files are not modified.

### 1 · Prepare the environment
Run once per new Colab runtime. This clones the public repository and installs the solver and YAML support.

In [ ]:
from pathlib import Path
import subprocess
import sys
import shutil

REPO = Path('/content/continuous-section-field')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/giovanniboscu/continuous-section-field.git', str(REPO)], check=True)
else:
    # Pick up files published after this Colab runtime first cloned the repo.
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'pypardiso', 'pyyaml'], check=True)
QUICKSTART = REPO / 'cuf' / 'quickstart'
assert (QUICKSTART / 'models/rectangular_prismatic_csf.yaml').is_file()
assert (QUICKSTART / 'problems/_halfwave.yaml').is_file()
assert (QUICKSTART / 'cases/_halfwave_legendre_N02.yaml').is_file()
assert (QUICKSTART / 'models/actions_colab.yaml').is_file(), 'Publish actions_colab.yaml to cuf/quickstart/models/ on GitHub.'
print('CSF-CUF ready. Choose parameters below and press Run.')

Change the parameters and press **▶**. The notebook updates temporary YAML files, uses **csf-actions** to display the CSF model, solves CUF, and generates stress maps at the selected x/L position.


**Physical problem : origin of loads and boundary conditions:** [Problem YAML](https://github.com/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/problems/_halfwave.yaml)

The problem YAML selects the Python adapter responsible for applying the loads and boundary conditions.

### Physical problem - torsion half-wave

![Torsion half-wave static scheme](https://raw.githubusercontent.com/giovanniboscu/continuous-section-field/main/cuf/quickstart/img/torsion_halfwave_static_scheme.jpg)

*Fixed-fixed rectangular beam with two opposite sinusoidal line loads acting in the global z direction at diagonally opposite section vertices.*

[Physical problem definition](https://github.com/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/problems/torsion_halfwave.yaml)


In [ ]:
# @title Configure model and run CSF-CUF { display-mode: "form" }
# @markdown **Numerical approximation** · [CUF order (case YAML)](https://github.com/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/cases/torsion_halfwave_legendre_N02.yaml)
cuf_order = 2 # @param {type:"integer"}
# @markdown [Longitudinal order (case YAML)](https://github.com/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/cases/torsion_halfwave_legendre_N02.yaml)
longitudinal_order = 6 # @param {type:"integer"}
# @markdown **Stress maps (transverse section)** · [Map generation (Python)](https://github.com/giovanniboscu/continuous-section-field/blob/main/src/csf/cuf/tools/plot_strain_stress_boundary_v10.py)
show_stress_maps = True # @param {type:"boolean"}
# @markdown [Section position x/L (map generator)](https://github.com/giovanniboscu/continuous-section-field/blob/main/src/csf/cuf/tools/plot_strain_stress_boundary_v10.py)
section_x_over_L = 0.50 # @param {type:"number"}
# @markdown **Material (Young's modulus, MPa)** · [Model YAML](https://github.com/giovanniboscu/continuous-section-field/blob/main/cuf/quickstart/models/rectangular_prismatic_csf.yaml)
E_S0 = 71700.0 # @param {type:"number"}
E_S1 = 71700.0 # @param {type:"number"}
# @markdown **Rectangular section dimensions (mm)**
width_S0 = 100.0 # @param {type:"number"}
height_S0 = 100.0 # @param {type:"number"}
width_S1 = 100.0 # @param {type:"number"}
height_S1 = 100.0 # @param {type:"number"}

from pathlib import Path
import subprocess
import sys
import shutil
import os
import yaml
from IPython.display import display, Markdown

if not (isinstance(cuf_order, int) and 1 <= cuf_order <= 20):
    raise ValueError('CUF order must be an integer from 1 to 20.')
if not (isinstance(longitudinal_order, int) and 1 <= longitudinal_order <= 20):
    raise ValueError('Longitudinal order must be an integer from 1 to 20.')
if not (0.0 <= float(section_x_over_L) <= 1.0):
    raise ValueError('Section position x/L must be between 0 and 1.')
for label, value in [('E_S0', E_S0), ('E_S1', E_S1),
                     ('width_S0', width_S0), ('height_S0', height_S0),
                     ('width_S1', width_S1), ('height_S1', height_S1)]:
    if not (0 < float(value) < float('inf')):
        raise ValueError(f'{label} must be positive and finite.')

# Work exclusively on disposable copies; preserve original relative YAML paths.
WORK = Path('/content/csf_cuf_quickstart_work')
for folder in ('models', 'problems', 'cases'):
    (WORK / folder).mkdir(parents=True, exist_ok=True)
for folder, filename in (
    ('models', 'rectangular_prismatic_csf.yaml'),
    ('problems', 'torsion_halfwave.yaml'),
    ('cases', 'torsion_halfwave_legendre_N02.yaml'),
):
    shutil.copy2(QUICKSTART / folder / filename, WORK / folder / filename)

model_path = WORK / 'models/rectangular_prismatic_csf.yaml'
case_path = WORK / 'cases/torsion_halfwave_legendre_N02.yaml'
model = yaml.safe_load(model_path.read_text(encoding='utf-8'))
case = yaml.safe_load(case_path.read_text(encoding='utf-8'))

def rectangle(width, height):
    w, h = width / 2.0, height / 2.0
    return [[-w, -h], [w, -h], [w, h], [-w, h]]

for station, E, width, height in (
    ('S0', E_S0, width_S0, height_S0),
    ('S1', E_S1, width_S1, height_S1),
):
    polygons = model['CSF']['sections'][station]['polygons']
    assert len(polygons) == 1, f'Expected one rectangle in {station}'
    polygons[0]['weight'] = float(E)
    polygons[0]['vertices'] = rectangle(float(width), float(height))

case['cuf']['order'] = cuf_order
case['longitudinal']['order'] = longitudinal_order
model_path.write_text(yaml.safe_dump(model, sort_keys=False), encoding='utf-8')
case_path.write_text(yaml.safe_dump(case, sort_keys=False), encoding='utf-8')

# Visualize the actual modified CSF model before solving.
action_source = QUICKSTART / 'models' / 'actions_colab.yaml'
if not action_source.is_file():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
if not action_source.is_file():
    raise FileNotFoundError(f'CSF actions configuration missing: {action_source}')
shutil.copy2(action_source, WORK / 'models' / 'actions_colab.yaml')
plot_dir = WORK / 'models' / 'out'
plot_dir.mkdir(parents=True, exist_ok=True)
for old in plot_dir.iterdir():
    if old.is_file() and old.suffix.lower() in ('.png', '.jpg', '.csv'):
        old.unlink()
plot_env = os.environ.copy()
plot_env['MPLBACKEND'] = 'Agg'
print('[CSF-ACTIONS] Starting model visualization...', flush=True)
plot_run = subprocess.run(
    ['csf-actions', 'rectangular_prismatic_csf.yaml', 'actions_colab.yaml'],
    cwd=WORK / 'models', env=plot_env, capture_output=True, text=True,
)
print('[CSF-ACTIONS] Exit code:', plot_run.returncode)
if plot_run.returncode != 0:
    print(plot_run.stdout)
    print(plot_run.stderr)
    raise RuntimeError('CSF actions failed')
print(f'Running CUF N{cuf_order:02d}; longitudinal order {longitudinal_order}')
print(f'E: S0={E_S0:g}, S1={E_S1:g} MPa')
print(f'Geometry: S0={width_S0:g} x {height_S0:g}, S1={width_S1:g} x {height_S1:g} mm')
print('Solving...')
run = subprocess.run(
    ['csf-cuf', 'cases/torsion_halfwave_legendre_N02.yaml'],
    cwd=WORK, capture_output=True, text=True,
)
if run.stdout.strip():
    print(run.stdout)
if run.returncode != 0:
    if run.stderr.strip():
        print(run.stderr)
    raise RuntimeError(f'csf-cuf failed (exit code {run.returncode})')
if run.stderr.strip():
    print(run.stderr)

# Direct CUF-only stress maps at a user-selected transverse station.
# This Quick Start model has L = 1000 mm (x in native mm).
if show_stress_maps:
    beam_length_mm = 1000.0
    section_x_mm = beam_length_mm * float(section_x_over_L)
    stress_dir = WORK / 'stress_maps'
    stress_dir.mkdir(parents=True, exist_ok=True)
    # Prevent images from previous parameter choices being shown again.
    for old_png in stress_dir.rglob('*.png'):
        old_png.unlink()
    print(f'[CUF MAPS] Generating stress maps at x/L={section_x_over_L:g} '
          f'(x={section_x_mm:g} mm)...', flush=True)
    stress_run = subprocess.run(
        [sys.executable, '-m', 'csf.cuf.tools.plot_strain_stress_boundary_v10',
         '--csf-yaml', str(model_path),
         '--cuf-output-root', str(WORK / 'output/torsion_halfwave_legendre_N02'),
         '--transverse-x', str(section_x_mm),
         '--transverse-grid', '100',
         '--output-dir', str(stress_dir),
         '--dpi', '150'],
        cwd=WORK, env=plot_env, capture_output=True, text=True,
    )
    if stress_run.returncode != 0:
        print(stress_run.stdout)
        print(stress_run.stderr)
        raise RuntimeError('CUF transverse stress map generation failed')
    stress_images = sorted(
        stress_dir.rglob('*.png'),
        key=lambda path: (
            'shear_stress_vector_transverse' not in path.stem.lower(),
            str(path),
        ),
    )
    display(Markdown(f'### CUF stress maps at x/L = {section_x_over_L:g} '
                     f'(x = {section_x_mm:g} mm)'))
    if not stress_images:
        print(stress_run.stdout)
        print('No PNG maps generated. Output directory:', stress_dir)
    for image_path in stress_images:
        display(Markdown('**' + image_path.stem.replace('_', ' ') + '**'))
        display(Image(filename=str(image_path)))
    print('[CUF MAPS] Figures:', len(stress_images), '| Output:', stress_dir)

from IPython.display import Image
sections = sorted(plot_dir.glob('sections_S0_S1*.png'))
properties = sorted(plot_dir.glob('section_properties*.png'))
materials = sorted(plot_dir.glob('material_variation*.png'))
for title, paths in (
    ('Sections S0 and S1', sections),
    ('Area and moments of inertia', properties),
    ('Material variation', materials),
):
    display(Markdown('### ' + title))
    if not paths:
        print('No images produced for:', title)
    for image_path in paths:
        display(Image(filename=str(image_path)))
csv_path = plot_dir / 'section_properties.csv'
display(Markdown('### Section properties at S0 and S1'))
if csv_path.is_file():
    print(csv_path.read_text(encoding='utf-8'))
else:
    print('CSV not generated:', csv_path)
print('[CSF-ACTIONS] Completed; images:', len(sections) + len(properties) + len(materials))


RESULTS = WORK / 'output/torsion_halfwave_legendre_N02'
response_path = RESULTS / 'response.txt'
if not response_path.is_file():
    raise FileNotFoundError(f'Solver completed but response.txt was not found: {response_path}')
response_text = response_path.read_text(encoding='utf-8')
display(Markdown('### Structural response (`response.txt`)'))
print(response_text)

# Keep previous response in this runtime for a simple side-by-side comparison.
previous = globals().get('_previous_csf_response')
if previous is not None:
    display(Markdown('### Previous run (for comparison)'))
    print(previous)
_previous_csf_response = response_text
print('Results:', RESULTS)
print('Original GitHub YAML files remain unchanged.')


**Done.** Change any parameter in step 2 and run it again. CSF plots, CUF stress maps, and numerical responses are regenerated. The original GitHub YAML files remain unchanged.
